# PubMedQA — Dataset Profile

This notebook describes the **PubMedQA PQA-L labeled dataset** used in the
Context Matters project.

The goal here is to understand the dataset itself:

- what kind of questions it contains;
- what information is available for each question;
- how the labels are distributed;
- how much supporting context is available;
- what biomedical topics occur frequently;
- why PubMedQA is useful for evaluating retrieval-augmented generation.

This notebook does **not** report Sprint-1, Sprint-2, or Sprint-3 model results.

## 1. Frozen Dataset Source

The project uses the exact frozen PubMedQA source and revision already defined
in the repository.

The selected configuration is the **PQA-L labeled subset**, containing
1,000 expert-labeled biomedical research questions.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

def display(obj):
    print(obj)

# Find repository root whether the notebook is launched from the repo root
# or from notebooks/datasets.
REPO = Path.cwd().resolve()

while REPO != REPO.parent and not (REPO / "scripts").is_dir():
    REPO = REPO.parent

assert (REPO / "scripts").is_dir(), "Could not locate repository root"

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("Repository:", REPO)

In [ ]:
from datasets import load_dataset, disable_progress_bar

from scripts.build_corpus_manifests import (
    PUBMEDQA_SOURCE,
    PUBMEDQA_CONFIG,
    PUBMEDQA_REVISION,
    PUBMEDQA_SPLIT,
    load_validated_pubmedqa_runtime_corpus,
)

disable_progress_bar()

dataset = load_dataset(
    PUBMEDQA_SOURCE,
    PUBMEDQA_CONFIG,
    split=PUBMEDQA_SPLIT,
    revision=PUBMEDQA_REVISION,
    download_mode="reuse_dataset_if_exists",
)

rows = tuple(dataset)

runtime_corpus = load_validated_pubmedqa_runtime_corpus(rows)
sample_manifest = runtime_corpus.sample_manifest
corpus_manifest = runtime_corpus.corpus_manifest

assert len(rows) == 1_000
assert len(sample_manifest.entries) == 1_000
assert corpus_manifest.document_count == 3_358

print("PubMedQA validation: PASS")
print(f"Questions:        {len(rows):,}")
print(f"Corpus passages:  {corpus_manifest.document_count:,}")
print(f"Source:           {PUBMEDQA_SOURCE}")
print(f"Configuration:    {PUBMEDQA_CONFIG}")
print(f"Split:            {PUBMEDQA_SPLIT}")
print(f"Frozen revision:  {PUBMEDQA_REVISION}")
print("LLM/API calls made: 0")

## 2. Dataset Schema

Each PubMedQA example contains five top-level fields.

| Field | Meaning |
|---|---|
| `pubid` | PubMed article identifier |
| `question` | Biomedical research question |
| `context` | Article-derived supporting information and metadata |
| `long_answer` | Human-written explanatory answer |
| `final_decision` | Gold categorical answer: `yes`, `no`, or `maybe` |

The nested `context` object additionally contains article text sections,
section labels, MeSH terms, and auxiliary prediction fields.

In [ ]:
schema = pd.DataFrame(
    [
        ["pubid", "integer", "PubMed article identifier"],
        ["question", "string", "Biomedical research question"],
        ["context.contexts", "list[string]", "Supporting article text sections"],
        ["context.labels", "list[string]", "Section labels for the context passages"],
        ["context.meshes", "list[string]", "Medical Subject Headings (MeSH)"],
        [
            "context.reasoning_required_pred",
            "list[string]",
            "Auxiliary dataset prediction field",
        ],
        [
            "context.reasoning_free_pred",
            "list[string]",
            "Auxiliary dataset prediction field",
        ],
        ["long_answer", "string", "Human-written explanatory answer"],
        ["final_decision", "string", "Gold yes / no / maybe decision"],
    ],
    columns=["Field", "Type", "Description"],
)

display(schema)

## 3. Example Question

The following example shows how one PubMedQA question combines:

1. a research question;
2. article-derived context;
3. a long explanatory answer;
4. a categorical gold decision.

In [ ]:
example = rows[0]

print("PubMed ID:")
print(example["pubid"])

print("\nQuestion:")
print(example["question"])

print("\nGold decision:")
print(example["final_decision"])

print("\nLong answer:")
print(example["long_answer"])

print("\nContext sections:")
for i, text in enumerate(example["context"]["contexts"], start=1):
    label = (
        example["context"]["labels"][i - 1]
        if i - 1 < len(example["context"]["labels"])
        else "UNKNOWN"
    )

    print(f"\n[{i}] {label}")
    print(text)

## 4. Gold Decision Distribution

PubMedQA is not simply a binary yes/no dataset.

The labeled subset contains three possible conclusions:

- **yes**
- **no**
- **maybe**

This makes the task useful for evaluating whether a system can distinguish
positive evidence, negative evidence, and genuinely uncertain findings.

In [ ]:
decision_counts = (
    pd.Series(
        [row["final_decision"] for row in rows],
        name="Decision",
    )
    .value_counts()
    .reindex(["yes", "no", "maybe"])
    .fillna(0)
    .astype(int)
)

decision_table = decision_counts.rename_axis("Decision").reset_index(
    name="Questions"
)

decision_table["Percent"] = (
    100 * decision_table["Questions"] / len(rows)
)

display(decision_table)

## 5. Supporting Context per Question

A PubMedQA question can contain multiple article-derived context sections.

This matters for RAG because the answer may depend on evidence distributed
across several parts of the associated research abstract.

In [ ]:
context_counts = pd.Series(
    [len(row["context"]["contexts"]) for row in rows],
    name="Context sections",
)

context_summary = context_counts.describe().to_frame().T

display(context_summary)

print(
    "Total raw context sections across the 1,000 questions:",
    f"{context_counts.sum():,}",
)
print(
    "Validated project corpus passages:",
    f"{corpus_manifest.document_count:,}",
)

## 6. Context Section Labels

PubMedQA context passages originate from structured biomedical abstracts.

The dataset preserves labels describing the role of individual sections,
such as background, methods, results, or conclusions when those labels are
available.

In [ ]:
section_labels = []

for row in rows:
    section_labels.extend(row["context"]["labels"])

section_label_counts = (
    pd.Series(section_labels, name="Section label")
    .value_counts()
    .rename_axis("Section label")
    .reset_index(name="Count")
)

display(section_label_counts)

## 7. Biomedical Topics — MeSH Terms

PubMedQA also contains **Medical Subject Headings (MeSH)** associated with
the underlying PubMed articles.

MeSH terms provide a compact view of the biomedical topics represented in
the labeled dataset.

In [ ]:
mesh_terms = []

for row in rows:
    mesh_terms.extend(row["context"]["meshes"])

mesh_counts = (
    pd.Series(mesh_terms, name="MeSH term")
    .value_counts()
    .rename_axis("MeSH term")
    .reset_index(name="Occurrences")
)

print("Unique MeSH terms:", f"{mesh_counts.shape[0]:,}")
display(mesh_counts.head(20))

## 8. Question and Answer Lengths

Text-length statistics help describe the type of reasoning task represented
by PubMedQA.

The questions are typically concise, while the reference long answers contain
substantially more explanatory content.

In [ ]:
lengths = pd.DataFrame(
    {
        "question_words": [
            len(row["question"].split())
            for row in rows
        ],
        "long_answer_words": [
            len(row["long_answer"].split())
            for row in rows
        ],
        "context_words": [
            sum(
                len(text.split())
                for text in row["context"]["contexts"]
            )
            for row in rows
        ],
    }
)

display(
    lengths.describe()
    .T[
        ["mean", "std", "min", "25%", "50%", "75%", "max"]
    ]
    .round(1)
)

## 9. One Example from Each Decision Class

Looking at one example from each class makes the `yes`, `no`, and `maybe`
semantics easier to understand.

In [ ]:
for decision in ["yes", "no", "maybe"]:
    row = next(
        row
        for row in rows
        if row["final_decision"] == decision
    )

    print("=" * 80)
    print("DECISION:", decision.upper())
    print("PubMed ID:", row["pubid"])
    print("Question:", row["question"])
    print("Long answer:", row["long_answer"])
    print()

## 10. Why PubMedQA Is Useful for This Project

PubMedQA provides a comparatively controlled biomedical QA setting.

It is useful for the Context Matters project because:

- questions have explicit gold decisions;
- supporting biomedical evidence is available;
- the same question can be evaluated with and without retrieved context;
- answer correctness has a clear dataset-native categorical metric;
- biomedical evidence can be technically detailed, so irrelevant retrieved
  passages can plausibly distract a language model.

This makes PubMedQA a useful baseline dataset before studying more complex
multi-hop and ambiguity-focused datasets.

## 11. Important Dataset Limitations

PubMedQA should not be interpreted as a general-purpose medical QA benchmark.

Important limitations include:

- the labeled PQA-L subset contains only 1,000 questions;
- questions originate from biomedical research articles rather than ordinary
  patient queries;
- categorical `yes/no/maybe` correctness does not capture every aspect of a
  generated explanation;
- article-derived context can make the task structurally different from
  open-domain retrieval;
- performance on PubMedQA alone does not establish general RAG performance.

These limitations are one reason the project also evaluates HotpotQA and ASQA.

## 12. Dataset Summary

In [ ]:
summary = pd.DataFrame(
    [
        ["Dataset", "PubMedQA"],
        ["Configuration", PUBMEDQA_CONFIG],
        ["Questions", f"{len(rows):,}"],
        ["Validated corpus passages", f"{corpus_manifest.document_count:,}"],
        ["Answer classes", "yes / no / maybe"],
        ["Primary text source", "Biomedical research abstracts"],
        ["Gold long answers", "Yes"],
        ["MeSH metadata", "Yes"],
        ["Role in project", "Controlled biomedical QA baseline"],
    ],
    columns=["Property", "Value"],
)

display(summary)

## 13. Reproducibility Check

In [ ]:
checks = pd.DataFrame(
    [
        ["Dataset contains exactly 1,000 questions", len(rows) == 1_000],
        [
            "Sample manifest contains exactly 1,000 entries",
            len(sample_manifest.entries) == 1_000,
        ],
        [
            "Validated corpus contains exactly 3,358 passages",
            corpus_manifest.document_count == 3_358,
        ],
        [
            "All decisions are yes/no/maybe",
            set(row["final_decision"] for row in rows)
            <= {"yes", "no", "maybe"},
        ],
        [
            "All questions are non-empty",
            all(bool(row["question"].strip()) for row in rows),
        ],
        [
            "All long answers are non-empty",
            all(bool(row["long_answer"].strip()) for row in rows),
        ],
    ],
    columns=["Check", "PASS"],
)

assert checks["PASS"].all(), checks[~checks["PASS"]]

display(checks)

print("PASS: PubMedQA dataset profile is reproducible")
print("LLM/API calls made by this notebook: 0")

## Reproducibility

This notebook uses the same frozen PubMedQA source, configuration, split,
revision, sample manifest, and corpus manifest as the main project.

It performs descriptive dataset analysis only and makes no LLM/API calls.